In [1]:
# Check whether easydiffraction is installed; install it if needed.
# Required for remote environments such as Google Colab.
import importlib.util

if importlib.util.find_spec('easydiffraction') is None:
    %pip install easydiffraction==2026.11.1 --extra-index-url https://download.easydiffraction.org/pypi/simple/


# Co₂SiO₄ on D20

&nbsp; &nbsp; &nbsp; &nbsp; &nbsp; &nbsp; 

This tutorial fits the olivine Co₂SiO₄ to constant-wavelength neutron powder data measured on
D20 at the ILL. Its structure and its measured pattern come from EasyDiffraction's collection;
the tutorial describes the experiment, links the two and fits the coordinates, the
displacements and the oxygen occupancies.

## Import the library

In [2]:
import easydiffraction as edi

## Define the project

In [3]:
project = edi.Project(name='cosio_d20')

## Define the structure

Load the structure from the collection. The CIF gives the space group P n m a, the cell and the
six sites, and marks the cell, coordinates, displacements and oxygen occupancies to fit. The fit
below frees them in stages, so start with all of them fixed.

In [4]:
structure_path = edi.download_data('pd-neut-cwl_cosio-d20-cosio', destination='data')
project.structures.add_from_cif_path(structure_path)
structure = project.structures['cosio']
for parameter in project.free_parameters:
    parameter.free = False

Data 'pd-neut-cwl_cosio-d20-cosio' is at data/pd-neut-cwl_cosio-d20-cosio.edi


In [5]:
project.display.structure('cosio')

View(html=<243,138 characters>, height=660)

## Define the experiment

The measured pattern also comes from the collection.

In [6]:
data_path = edi.download_data('pd-neut-cwl_cosio-d20-d20-data', destination='data')
project.experiments.add_from_data_path(
    name='d20',
    data_path=data_path,
    sample_form='powder',
    beam_mode='constant wavelength',
    radiation_probe='neutron',
    scattering_type='bragg',
)
experiment = project.experiments['d20']

Data 'pd-neut-cwl_cosio-d20-d20-data' is at data/pd-neut-cwl_cosio-d20-d20.xye


D20's wavelength is 1.87 Å. The profile is the Thompson-Cox-Hastings pseudo-Voigt, and a flat
background runs through fourteen points.

In [7]:
experiment.instrument.setup_wavelength = 1.87
experiment.instrument.calib_twotheta_offset = 0
experiment.peak.broad_gauss_u = 0.2
experiment.peak.broad_gauss_v = -0.5
experiment.peak.broad_gauss_w = 0.4
experiment.peak.broad_lorentz_y = 0.01
for position in (8, 9, 10, 11, 12, 15, 25, 30, 50, 70, 90, 110, 130, 150):
    experiment.background.create(position=position, intensity=340)
experiment.linked_structures.create(structure_id='cosio', scale=1)

In [8]:
project.display.pattern('d20')

View(html=<242,313 characters>, height=480)

## Perform the analysis

The fit goes in stages, freeing one group of parameters at a time.

### Scale, 2θ zero and cell

The scale sets the overall height of the pattern, and the zero shift and the three cell lengths
put the peaks where they are measured.

In [9]:
experiment.linked_structures['cosio'].scale.free = True
experiment.instrument.calib_twotheta_offset.free = True
structure.cell.length_a.free = True
structure.cell.length_b.free = True
structure.cell.length_c.free = True

In [10]:
project.display.parameters.free()

#,datablock,category,entry,parameter,value,s.u.,min,max,units
1,cosio,cell,,length_a,10.3,,0,30,Å
2,cosio,cell,,length_b,6,,0,30,Å
3,cosio,cell,,length_c,4.8,,0,30,Å
4,d20,instrument,,calib_twotheta_offset,0,0,-inf,inf,deg
5,d20,linked_structure,cosio,scale,1,0,-inf,inf,


In [11]:
project.analysis.fit()
project.display.fit.results()

edi fit: project cosio_d20, 1418 pts fitted / 1418 loaded, 5 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                       40.51       452.01
      1      0.026     20.61       117.03   74.1% ↓
      2      0.030     20.10       111.27   4.9% ↓
      3      0.033     20.09       111.14   0.1% ↓
      4      0.037     20.09       111.14   
      5      0.041     20.09       111.14   
      6      0.046     20.09       111.14   
  done · converged · 6 iters · reduced χ² 111.14 · Rwp 20.09 % · 0.05 s


View(html=<85,126 characters>, height=376)

In [12]:
project.display.pattern('d20')

View(html=<243,154 characters>, height=480)

### Background

With the peaks in place, the fourteen background points follow the level between them.

In [13]:
for point in experiment.background:
    point.intensity.free = True

In [14]:
project.analysis.fit()
project.display.fit.results()

edi fit: project cosio_d20, 1418 pts fitted / 1418 loaded, 19 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                       20.09       112.25
      1      0.032      5.51         8.45   92.5% ↓
  done · converged · 1 iters · reduced χ² 8.45 · Rwp 5.51 % · 0.04 s


View(html=<102,395 characters>, height=376)

In [15]:
project.display.pattern('d20')

View(html=<256,671 characters>, height=480)

### Peak widths

The profile terms now fit the widths and shapes of the peaks. diffraction-lib's tutorial also
fits a peak asymmetry; this case's profile has none.

In [16]:
experiment.peak.broad_gauss_u.free = True
experiment.peak.broad_gauss_v.free = True
experiment.peak.broad_gauss_w.free = True
experiment.peak.broad_lorentz_y.free = True

In [17]:
project.analysis.fit()
project.display.fit.results()

edi fit: project cosio_d20, 1418 pts fitted / 1418 loaded, 23 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                        5.51         8.47
      1      0.032      4.80         6.42   24.2% ↓
      2      0.037      4.80         6.42   
      3      0.044      4.80         6.42   
  done · converged · 3 iters · reduced χ² 6.42 · Rwp 4.80 % · 0.05 s


View(html=<106,697 characters>, height=376)

In [18]:
project.display.pattern('d20')

View(html=<256,894 characters>, height=480)

### Atom positions

The peak intensities now depend mostly on where the atoms are. Co1 sits on a centre of
symmetry and the other sites on the mirror plane, so only their free coordinates are fitted.

In [19]:
for site in structure.atom_sites:
    for coordinate in (site.fract_x, site.fract_y, site.fract_z):
        coordinate.free = not coordinate.symmetry_constrained

In [20]:
project.analysis.fit()
project.display.fit.results()

edi fit: project cosio_d20, 1418 pts fitted / 1418 loaded, 34 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                        4.80         6.47
      1      0.040      4.14         4.82   25.5% ↓
      2      0.048      4.14         4.82   
  done · converged · 2 iters · reduced χ² 4.82 · Rwp 4.14 % · 0.05 s


View(html=<122,251 characters>, height=376)

In [21]:
project.display.pattern('d20')

View(html=<256,910 characters>, height=480)

### Displacements

The displacements change the intensities more at high angle.

In [22]:
for site in structure.atom_sites:
    site.adp_iso.free = True

In [23]:
project.analysis.fit()
project.display.fit.results()

edi fit: project cosio_d20, 1418 pts fitted / 1418 loaded, 40 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                        4.14         4.84
      1      0.045      3.99         4.49   7.2% ↓
  done · converged · 1 iters · reduced χ² 4.49 · Rwp 3.99 % · 0.05 s


View(html=<128,569 characters>, height=376)

In [24]:
project.display.pattern('d20')

View(html=<256,965 characters>, height=480)

In [25]:
project.display.fit.correlations()

,1,2,3,4,5
1 cosio.cell.length_b,,,,,
2 d20.instrument.calib_twotheta_offset,0.66,,,,
3 d20.peak.broad_gauss_u,0.04,0.03,,,
4 d20.peak.broad_gauss_v,-0.02,-0.03,-0.95,,
5 d20.peak.broad_gauss_w,0.00,0.03,0.86,-0.93,


### Constrain the two Co sites

The correlations show no strong pair for the two Co sites, yet the fit gives them very different
displacements, Co1's even negative, which no atom can have: both are Co and scatter alike, so
the pattern cannot tell their thermal motions apart. Give each an alias and constrain Co2's to
follow Co1's, starting again from 0.5 Å², then fit again; the shared displacement is positive and
its s.u. smaller than either had alone.

In [26]:
structure.atom_sites['Co2'].adp_iso.free = False
structure.atom_sites['Co1'].adp_iso = 0.5
project.analysis.aliases.create(id='biso_Co1', param=structure.atom_sites['Co1'].adp_iso)
project.analysis.aliases.create(id='biso_Co2', param=structure.atom_sites['Co2'].adp_iso)
project.analysis.constraints.create(id='biso_Co2', expression='biso_Co2 = biso_Co1')
project.display.constraints()

View(html=<77,884 characters>, height=180)

In [27]:
project.analysis.fit()
project.display.fit.results()

edi fit: project cosio_d20, 1418 pts fitted / 1418 loaded, 39 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                        4.05         4.64
      1      0.044      4.02         4.57   1.6% ↓
  done · converged · 1 iters · reduced χ² 4.57 · Rwp 4.02 % · 0.05 s


View(html=<125,760 characters>, height=376)

In [28]:
project.display.pattern('d20')

View(html=<257,396 characters>, height=480)

### Oxygen occupancies

Last, the oxygen occupancies, which change the intensities as the displacements do.

In [29]:
structure.atom_sites['O1'].occupancy.free = True
structure.atom_sites['O2'].occupancy.free = True
structure.atom_sites['O3'].occupancy.free = True

In [30]:
project.analysis.fit()
project.display.fit.results()

edi fit: project cosio_d20, 1418 pts fitted / 1418 loaded, 42 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                        4.02         4.58
      1      0.041      3.97         4.45   2.7% ↓
  done · converged · 1 iters · reduced χ² 4.45 · Rwp 3.97 % · 0.05 s


View(html=<130,418 characters>, height=376)

In [31]:
project.display.pattern('d20')

View(html=<257,406 characters>, height=480)

In [32]:
project.display.pattern('d20', x_min=40, x_max=60)

View(html=<257,438 characters>, height=480)

In [33]:
project.display.fit.correlations()

,1,2,3,4,5
1 cosio.atom_site.O3.occupancy,,,,,
2 d20.linked_structure.cosio.scale,-0.79,,,,
3 d20.peak.broad_gauss_u,0.04,-0.06,,,
4 d20.peak.broad_gauss_v,-0.02,-0.01,-0.95,,
5 d20.peak.broad_gauss_w,-0.02,0.04,0.85,-0.93,


In [34]:
project.display.structure('cosio')

View(html=<240,879 characters>, height=660)

## Save the project

In [35]:
project.save_as('cosio_d20')

Saving project to 'cosio_d20'
